# Testes de Fusão Texto × Série Temporal (Colab)

Ablação com deep learning seguindo a proposta do paper (`docs/plan.md` §2.1, §2.3, §3.1, §3.3, §4):

1. **TS-only** — só embeddings MOMENT da série temporal
2. **Text-only** — só embeddings de texto (lag 1 dia)
3. **Fusão** — cross-attention bidirecional (`TemporalFusion`)

Métricas: RMSE/MAE, acurácia de sinal, AUC, IC, backtest vs buy & hold, métricas por regime de volatilidade, entropia de atenção por regime, efeito causal do texto (shuffle) e análise lead-lag.

**Pré-requisitos:**

1. Ativar GPU em *Runtime → Change runtime type → GPU*
2. Gerar o painel uma vez no PC (só CPU, ~30s): `uv run python scripts/export_fusion_panel.py`
3. Enviar `data/fusion_panel.npz` para o Google Drive na pasta `MyDrive/latent_fusion_data/`
4. O código é clonado do GitHub; se o repo for privado, copie a pasta do repo para `MyDrive/latent_fusion/`

In [ ]:
import subprocess
import sys
from pathlib import Path

IN_COLAB = False
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    pass

if IN_COLAB:
    drive.mount("/content/drive")
    REPO_ROOT = Path("/content/latent_fusion")
    if not REPO_ROOT.exists():
        subprocess.run(["git", "clone", "https://github.com/rbfcdog/latent_fusion.git", str(REPO_ROOT)], capture_output=True)
    if not (REPO_ROOT / "src" / "fusion").exists():
        REPO_ROOT = Path("/content/drive/MyDrive/latent_fusion")
    PANEL_PATH = Path("/content/drive/MyDrive/latent_fusion_data/fusion_panel.npz")
    OUT_DIR = Path("/content/drive/MyDrive/latent_fusion_data/outputs")
else:
    REPO_ROOT = Path.cwd()
    while not ((REPO_ROOT / "src").exists() and (REPO_ROOT / "pyproject.toml").exists()) and REPO_ROOT != REPO_ROOT.parent:
        REPO_ROOT = REPO_ROOT.parent
    PANEL_PATH = REPO_ROOT / "data" / "fusion_panel.npz"
    OUT_DIR = REPO_ROOT / "outputs" / "fusion"

OUT_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(REPO_ROOT))

import torch

from src.fusion.dataset import load_panel_file
from src.fusion.run_tests import run_ablation, plot_equity, plot_lead_lag

{
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "repo": str(REPO_ROOT),
    "panel_encontrado": PANEL_PATH.exists(),
}

In [ ]:
panel = load_panel_file(PANEL_PATH)

{
    "tickers": len(panel["tickers"]),
    "treino": len(panel["splits"]["train"]),
    "validação": len(panel["splits"]["val"]),
    "teste": len(panel["splits"]["test"]),
}

In [ ]:
result = run_ablation(
    panel,
    device="cuda" if torch.cuda.is_available() else "cpu",
    out_dir=OUT_DIR,
)

result["history"]

## Ablação (paper §2.1): unimodal vs multimodal

Estratégia: `sign(pred)` equal-weight vs buy & hold, avaliada só no teste temporal.

In [ ]:
result["table"]

## Entropia de atenção por regime (§3.1) e efeito causal do texto (§3.3)

In [ ]:
result["attn_summary"]

## Lead-lag texto × preço (§2.3)

Correlação entre drift de embedding textual e retorno futuro em cada lag (média de 49 tickers).

In [ ]:
result["lead_lag"]

## Equity curves no teste

In [ ]:
plot_equity(result["dailies"], OUT_DIR / "equity_curves.png")

In [ ]:
plot_lead_lag(result["lead_lag"], OUT_DIR / "lead_lag.png")

In [ ]:
result["table"].to_csv(OUT_DIR / "results.csv", index=False)
result["lead_lag"].to_csv(OUT_DIR / "lead_lag.csv", index=False)

{"salvo_em": str(OUT_DIR)}